# 03 - Retail Analytics and Agent Actions
ينشئ traffic وqueue history وanomaly alerts لكل مطعم/مكان بشكل مستقل، ثم يحفظها CSV وDuckDB.

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLES_DIR = PROJECT_ROOT / 'Output' / 'tables'
DATABASE_DIR = PROJECT_ROOT / 'Output' / 'database'
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

QUARTER_LABELS_AR = {
    1: 'الربع الأول',
    2: 'الربع الثاني',
    3: 'الربع الثالث',
    4: 'الربع الرابع',
}


def format_video_time(seconds):
    seconds = max(0, int(round(seconds)))
    return f'{seconds // 60:02d}:{seconds % 60:02d}'


def build_video_quarter_counts(events, video_metadata):
    """Count unique camera-local tracks in each quarter of every camera video."""
    required_event_columns = {'camera_id', 'frame_index', 'camera_track_uid'}
    required_metadata_columns = {'camera_id', 'fps', 'total_frames', 'duration_sec'}
    if not required_event_columns.issubset(events.columns):
        missing = sorted(required_event_columns.difference(events.columns))
        raise ValueError(f'zone_events.csv is missing: {missing}')
    if not required_metadata_columns.issubset(video_metadata.columns):
        missing = sorted(required_metadata_columns.difference(video_metadata.columns))
        raise ValueError(f'video_metadata.csv is missing: {missing}')

    customer_events = events.copy()
    if 'is_customer' in customer_events.columns:
        customer_events = customer_events[
            customer_events['is_customer'].astype(str).str.lower().isin({'true', '1'})
        ]

    metadata_columns = ['camera_id', 'fps', 'total_frames', 'duration_sec']
    quarter_events = customer_events.merge(
        video_metadata[metadata_columns].drop_duplicates('camera_id'),
        on='camera_id',
        how='inner',
    )
    quarter_events = quarter_events[quarter_events['total_frames'].gt(0)].copy()
    if quarter_events.empty:
        return pd.DataFrame()

    quarter_events['video_quarter'] = np.minimum(
        (quarter_events['frame_index'] * 4 / quarter_events['total_frames']).astype(int) + 1,
        4,
    )
    group_columns = ['store_id', 'camera_id', 'video_quarter', 'fps', 'total_frames', 'duration_sec']
    quarter_counts = (
        quarter_events.groupby(group_columns, as_index=False)
        .agg(
            unique_camera_tracks=('camera_track_uid', 'nunique'),
            observations=('camera_track_uid', 'size'),
        )
        .sort_values(['store_id', 'camera_id', 'video_quarter'])
    )

    quarter_index = quarter_counts['video_quarter']
    total_frames = quarter_counts['total_frames'].astype(int)
    quarter_counts['quarter_label_ar'] = quarter_index.map(QUARTER_LABELS_AR)
    quarter_counts['quarter_start_frame'] = ((quarter_index - 1) * total_frames // 4).astype(int)
    quarter_counts['quarter_end_frame'] = (
        (quarter_index * total_frames + 3) // 4 - 1
    ).clip(upper=total_frames - 1).astype(int)
    quarter_counts['quarter_start_sec'] = quarter_counts['quarter_start_frame'] / quarter_counts['fps']
    quarter_counts['quarter_end_sec'] = (quarter_counts['quarter_end_frame'] + 1) / quarter_counts['fps']
    quarter_counts['frame_range'] = quarter_counts.apply(
        lambda row: f"{row.quarter_start_frame:,}–{row.quarter_end_frame:,}",
        axis=1,
    )
    quarter_counts['time_range'] = quarter_counts.apply(
        lambda row: f"{format_video_time(row.quarter_start_sec)}–{format_video_time(row.quarter_end_sec)}",
        axis=1,
    )
    return quarter_counts


def prepare_camera_customer_events(events):
    """Return valid customer events, ordered inside each individual camera."""
    required_columns = {
        'store_id', 'camera_id', 'camera_track_uid', 'zone_id',
        'floor_x', 'floor_y', 'timestamp_sec', 'frame_index',
    }
    missing = sorted(required_columns.difference(events.columns))
    if missing:
        raise ValueError(f'zone_events.csv is missing movement columns: {missing}')

    customer_events = events.copy()
    if 'is_customer' in customer_events.columns:
        customer_events = customer_events[
            customer_events['is_customer'].astype(str).str.lower().isin({'true', '1'})
        ]
    elif 'is_employee' in customer_events.columns:
        customer_events = customer_events[~customer_events['is_employee'].astype(bool)]

    customer_events = customer_events[customer_events['zone_id'].notna()].copy()
    customer_events = customer_events[customer_events['zone_id'].ne('outside')]
    for column in ('floor_x', 'floor_y', 'timestamp_sec', 'frame_index'):
        customer_events[column] = pd.to_numeric(customer_events[column], errors='coerce')
    customer_events = customer_events.dropna(
        subset=['floor_x', 'floor_y', 'timestamp_sec', 'frame_index']
    ).copy()

    # Keeping camera_id in the key prevents movement from bridging cameras.
    return customer_events.sort_values(
        ['store_id', 'camera_id', 'camera_track_uid', 'timestamp_sec', 'frame_index']
    ).reset_index(drop=True)



def build_movement_metrics(events):
    """Calculate distance and per-zone dwell time for each camera-local track."""
    points = prepare_camera_customer_events(events)
    output_columns = [
        'store_id', 'camera_id', 'camera_track_uid', 'zone_id', 'zone_label_ar',
        'total_distance', 'dwell_time_per_zone', 'point_count',
        'track_start_sec', 'track_end_sec',
    ]
    if points.empty:
        return pd.DataFrame(columns=output_columns)

    track_keys = ['store_id', 'camera_id', 'camera_track_uid']
    previous_xy = points.groupby(track_keys, sort=False)[['floor_x', 'floor_y']].shift()
    points['segment_distance'] = np.hypot(
        points['floor_x'] - previous_xy['floor_x'],
        points['floor_y'] - previous_xy['floor_y'],
    ).fillna(0.0)
    next_timestamp = points.groupby(track_keys, sort=False)['timestamp_sec'].shift(-1)
    points['dwell_seconds'] = (
        next_timestamp - points['timestamp_sec']
    ).clip(lower=0).fillna(0.0)

    track_totals = (
        points.groupby(track_keys, as_index=False)
        .agg(
            total_distance=('segment_distance', 'sum'),
            point_count=('camera_track_uid', 'size'),
            track_start_sec=('timestamp_sec', 'min'),
            track_end_sec=('timestamp_sec', 'max'),
        )
    )
    if 'zone_label_ar' not in points.columns:
        points['zone_label_ar'] = points['zone_id']
    zone_dwell = (
        points.groupby(track_keys + ['zone_id', 'zone_label_ar'], as_index=False)
        .agg(dwell_time_per_zone=('dwell_seconds', 'sum'))
    )
    movement_metrics = zone_dwell.merge(track_totals, on=track_keys, how='left')
    for column in ('total_distance', 'dwell_time_per_zone', 'track_start_sec', 'track_end_sec'):
        movement_metrics[column] = movement_metrics[column].round(3)
    return movement_metrics[output_columns].sort_values(
        track_keys + ['zone_id']
    ).reset_index(drop=True)


def build_zone_transitions(events):
    """Count distinct camera-local customers moving between consecutive zones."""
    points = prepare_camera_customer_events(events)
    output_columns = ['store_id', 'camera_id', 'from_zone_id', 'to_zone_id', 'transition_count']
    if points.empty:
        return pd.DataFrame(columns=output_columns)

    track_keys = ['store_id', 'camera_id', 'camera_track_uid']
    points['from_zone_id'] = points.groupby(track_keys, sort=False)['zone_id'].shift()
    changed_zone = points[
        points['from_zone_id'].notna()
        & points['zone_id'].ne(points['from_zone_id'])
    ].copy()
    if changed_zone.empty:
        return pd.DataFrame(columns=output_columns)

    # A person contributes once per directed pair, even if they revisit it later.
    person_transitions = changed_zone.rename(columns={'zone_id': 'to_zone_id'})[
        track_keys + ['from_zone_id', 'to_zone_id']
    ].drop_duplicates()
    transitions = (
        person_transitions.groupby(
            ['store_id', 'camera_id', 'from_zone_id', 'to_zone_id'], as_index=False
        )
        .size()
        .rename(columns={'size': 'transition_count'})
        .sort_values(['store_id', 'camera_id', 'transition_count'], ascending=[True, True, False])
        .reset_index(drop=True)
    )
    return transitions[output_columns]


events = pd.read_csv(TABLES_DIR / 'zone_events.csv')
if events.empty:
    raise ValueError('zone_events.csv is empty. Check zones/calibration and rerun notebook 02.')
if 'store_id' not in events.columns:
    events['store_id'] = 'default_store'

metadata_path = TABLES_DIR / 'video_metadata.csv'
if not metadata_path.exists():
    raise FileNotFoundError('video_metadata.csv is missing. Rerun notebook 01 to capture FPS and total frames.')
video_metadata = pd.read_csv(metadata_path)
video_quarter_counts = build_video_quarter_counts(events, video_metadata)
movement_metrics = build_movement_metrics(events)
zone_transitions = build_zone_transitions(events)

WINDOW_SECONDS = 60
QUEUE_ALERT_THRESHOLD = 5
events['window_start_sec'] = (
    events['timestamp_sec'] // WINDOW_SECONDS * WINDOW_SECONDS
).astype(int)

group_columns = [
    'store_id', 'camera_id', 'window_start_sec', 'zone_id', 'zone_label_ar', 'zone_kind'
]
traffic = (
    events.groupby(group_columns, as_index=False)
    .agg(
        customer_count=('camera_track_uid', 'nunique'),
        observations=('camera_track_uid', 'size'),
    )
)
traffic['window_start'] = pd.to_timedelta(
    traffic['window_start_sec'], unit='s'
).astype(str)

baseline = (
    traffic.groupby(['store_id', 'camera_id', 'zone_id'])['customer_count']
    .agg(['mean', 'std'])
    .reset_index()
)
traffic = traffic.merge(baseline, on=['store_id', 'camera_id', 'zone_id'], how='left')
traffic['z_score'] = np.where(
    traffic['std'].fillna(0) > 0,
    (traffic['customer_count'] - traffic['mean']) / traffic['std'],
    0.0,
)
traffic['is_anomaly'] = traffic['z_score'].abs() >= 2.0
traffic = traffic.drop(columns=['mean', 'std'])

queue_history = traffic[traffic['zone_kind'].eq('queue')].copy()
queue_columns = [
    'store_id', 'camera_id', 'window_start_sec', 'window_start', 'zone_id',
    'zone_label_ar', 'queue_length', 'predicted_queue_next_window'
]
if queue_history.empty:
    queue_history = pd.DataFrame(columns=queue_columns)
else:
    queue_history = queue_history.rename(columns={'customer_count': 'queue_length'})
    queue_history = queue_history.sort_values(
        ['store_id', 'camera_id', 'zone_id', 'window_start_sec']
    )
    queue_change = (
        queue_history.groupby(['store_id', 'camera_id', 'zone_id'])['queue_length']
        .diff()
        .fillna(0)
    )
    queue_history['predicted_queue_next_window'] = (
        queue_history['queue_length'] + queue_change
    ).clip(lower=0).round().astype(int)
    queue_history = queue_history[queue_columns]

actions = []
for row in traffic[traffic['is_anomaly']].itertuples():
    actions.append({
        'store_id': row.store_id,
        'camera_id': row.camera_id,
        'created_at': row.window_start,
        'action_type': 'crowd_anomaly',
        'severity': 'high',
        'zone_id': row.zone_id,
        'message_ar': (
            f'ازدحام غير طبيعي في {row.zone_label_ar}: '
            f'{row.customer_count} عميل.'
        ),
    })

long_queues = queue_history[
    queue_history['predicted_queue_next_window'] >= QUEUE_ALERT_THRESHOLD
]
for row in long_queues.itertuples():
    actions.append({
        'store_id': row.store_id,
        'camera_id': row.camera_id,
        'created_at': row.window_start,
        'action_type': 'queue_prediction',
        'severity': 'medium',
        'zone_id': row.zone_id,
        'message_ar': (
            f'توقع طابور طويل عند {row.zone_label_ar}: '
            f'{row.predicted_queue_next_window} عملاء. '
            'اقترح فتح كاشير إضافي.'
        ),
    })

action_columns = [
    'store_id', 'camera_id', 'created_at', 'action_type', 'severity', 'zone_id', 'message_ar'
]
agent_actions = pd.DataFrame(actions, columns=action_columns)

traffic.to_csv(TABLES_DIR / 'zone_traffic.csv', index=False)
queue_history.to_csv(TABLES_DIR / 'queue_history.csv', index=False)
agent_actions.to_csv(TABLES_DIR / 'agent_actions.csv', index=False)
video_quarter_counts.to_csv(TABLES_DIR / 'video_quarter_counts.csv', index=False)
movement_metrics.to_csv(TABLES_DIR / 'movement_metrics.csv', index=False)
zone_transitions.to_csv(TABLES_DIR / 'zone_transitions.csv', index=False)

db_path = DATABASE_DIR / 'retail_intelligence.duckdb'
with duckdb.connect(str(db_path)) as connection:
    connection.register('traffic_df', traffic)
    connection.register('queue_df', queue_history)
    connection.register('actions_df', agent_actions)
    connection.register('video_quarters_df', video_quarter_counts)
    connection.register('movement_metrics_df', movement_metrics)
    connection.register('zone_transitions_df', zone_transitions)
    connection.execute('CREATE OR REPLACE TABLE zone_traffic AS SELECT * FROM traffic_df')
    connection.execute('CREATE OR REPLACE TABLE queue_history AS SELECT * FROM queue_df')
    connection.execute('CREATE OR REPLACE TABLE agent_actions AS SELECT * FROM actions_df')
    connection.execute('CREATE OR REPLACE TABLE video_quarter_counts AS SELECT * FROM video_quarters_df')
    connection.execute('CREATE OR REPLACE TABLE movement_metrics AS SELECT * FROM movement_metrics_df')
    connection.execute('CREATE OR REPLACE TABLE zone_transitions AS SELECT * FROM zone_transitions_df')

print(f'Stores/places analyzed: {traffic.store_id.nunique():,}')
print(f'Video quarter counts: {len(video_quarter_counts):,}')
print(f'Movement metric rows: {len(movement_metrics):,}')
print(f'Zone transitions: {len(zone_transitions):,}')
print(f'Saved analytics to {db_path.relative_to(PROJECT_ROOT)}')
display(traffic.head())
display(video_quarter_counts.head())
display(movement_metrics.head())
display(zone_transitions.head())
display(agent_actions.head())

Stores/places analyzed: 1
Video quarter counts: 16
Movement metric rows: 458
Zone transitions: 7
Saved analytics to Output\database\retail_intelligence.duckdb


,store_id,camera_id,window_start_sec,zone_id,zone_label_ar,zone_kind,customer_count,observations,window_start,z_score,is_anomaly
0,place_05,CAFE_place_05_camera_17_15min,0,outside,outside,outside,12,1058,0 days 00:00:00,-1.251442,False
1,place_05,CAFE_place_05_camera_17_15min,0,shelves,الأرفف,sales,10,527,0 days 00:00:00,-0.415771,False
2,place_05,CAFE_place_05_camera_17_15min,60,outside,outside,outside,13,1045,0 days 00:01:00,-1.121983,False
3,place_05,CAFE_place_05_camera_17_15min,60,shelves,الأرفف,sales,9,559,0 days 00:01:00,-0.646756,False
4,place_05,CAFE_place_05_camera_17_15min,120,outside,outside,outside,30,1026,0 days 00:02:00,1.078830,False


,store_id,camera_id,video_quarter,fps,total_frames,duration_sec,unique_camera_tracks,observations,quarter_label_ar,quarter_start_frame,quarter_end_frame,quarter_start_sec,quarter_end_sec,frame_range,time_range
0,place_05,CAFE_place_05_camera_17_15min,1,5.0,4500,900.0,69,5759,الربع الأول,0,1124,0.0,225.0,"0–1,124",00:00–03:45
1,place_05,CAFE_place_05_camera_17_15min,2,5.0,4500,900.0,67,5895,الربع الثاني,1125,2249,225.0,450.0,"1,125–2,249",03:45–07:30
2,place_05,CAFE_place_05_camera_17_15min,3,5.0,4500,900.0,83,5565,الربع الثالث,2250,3374,450.0,675.0,"2,250–3,374",07:30–11:15
3,place_05,CAFE_place_05_camera_17_15min,4,5.0,4500,900.0,120,5282,الربع الرابع,3375,4499,675.0,900.0,"3,375–4,499",11:15–15:00
4,place_05,CAFE_place_05_camera_18_15min,1,5.0,4500,900.0,70,5965,الربع الأول,0,1124,0.0,225.0,"0–1,124",00:00–03:45


,store_id,camera_id,camera_track_uid,zone_id,zone_label_ar,total_distance,dwell_time_per_zone,point_count,track_start_sec,track_end_sec
0,place_05,CAFE_place_05_camera_17_15min,place_05::CAFE_place_05_camera_17_15min::10,shelves,الأرفف,13.476,0.4,2,0.0,0.4
1,place_05,CAFE_place_05_camera_17_15min,place_05::CAFE_place_05_camera_17_15min::1000,shelves,الأرفف,14.760,0.4,2,728.0,728.4
2,place_05,CAFE_place_05_camera_17_15min,place_05::CAFE_place_05_camera_17_15min::1003,shelves,الأرفف,78.182,0.4,2,728.8,729.2
3,place_05,CAFE_place_05_camera_17_15min,place_05::CAFE_place_05_camera_17_15min::1010,shelves,الأرفف,791.464,21.6,43,730.8,752.4
4,place_05,CAFE_place_05_camera_17_15min,place_05::CAFE_place_05_camera_17_15min::1013,shelves,الأرفف,137.956,7.2,14,732.0,739.2


,store_id,camera_id,from_zone_id,to_zone_id,transition_count
0,place_05,CAFE_place_05_camera_17_15min,shelves,checkout,1
1,place_05,CAFE_place_05_camera_19_15min,shelves,checkout,31
2,place_05,CAFE_place_05_camera_19_15min,checkout,shelves,27
3,place_05,CAFE_place_05_camera_19_15min,shelves,entrance,12
4,place_05,CAFE_place_05_camera_19_15min,entrance,shelves,10


,store_id,camera_id,created_at,action_type,severity,zone_id,message_ar
0,place_05,CAFE_place_05_camera_18_15min,0 days 00:02:00,crowd_anomaly,high,outside,ازدحام غير طبيعي في outside: 39 عميل.
1,place_05,CAFE_place_05_camera_19_15min,0 days 00:12:00,crowd_anomaly,high,checkout,ازدحام غير طبيعي في الكاشير: 8 عميل.
2,place_05,CAFE_place_05_camera_20_15min,0 days 00:12:00,crowd_anomaly,high,outside,ازدحام غير طبيعي في outside: 36 عميل.
3,place_05,CAFE_place_05_camera_19_15min,0 days 00:01:00,queue_prediction,medium,checkout,توقع طابور طويل عند الكاشير: 5 عملاء. اقترح فت...
4,place_05,CAFE_place_05_camera_19_15min,0 days 00:02:00,queue_prediction,medium,checkout,توقع طابور طويل عند الكاشير: 5 عملاء. اقترح فت...


التنبيهات محفوظة محليًا فقط في هذه النسخة. تكامل Telegram أو WhatsApp يحتاج credentials وموافقة صريحة على قناة الإرسال.